# 04 Build models

**Question:** using only what a team knows on draft night, how well can I predict a player's early career?

My original version of this project reached R² = 0.94, but it used career minutes, which are only known
*after* a career. Here I only use facts available on draft night, and I compare every model with a simple baseline.

In [2]:
import pandas as pd
import numpy as np 

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier

LAST_TRAIN_YEAR = 2010

## Features

- **`log_pick`**: the natural log of the pick number. Notebook 03 showed value drops fast for early picks and slowly
  for late ones. The log stretches out the top picks and squeezes the late ones together, so a straight-line model
  can follow that curve.
- **`no_college`**: 1 if the player had no college listed, otherwise 0. Models need numbers, not text.

In [17]:
players = pd.read_csv("../data/processed/draft_players_targets.csv")

players["log_pick"] = np.log(players["overall_pick"])
players["no_college"] = (players["background"] == "no_college").astype(int)



## Split by draft year

- **Train:** drafts 1989–2010 (1,262 picks). The models learn from these.
- **Test:** drafts 2011–2018 (480 picks). The models never see these while learning; I grade them here.

I split by time instead of randomly because a team can only learn from past drafts. This copies real use
and stops information from later drafts leaking into training.

In [4]:
train = players[players["draft_year"] <= LAST_TRAIN_YEAR].copy()
test = players[players["draft_year"] > LAST_TRAIN_YEAR].copy()

print(len(train))
print(len(test))


1262
480


## Baseline: always guess the average

Before any model, I check a "dumb" prediction: give every test player the training average (6.14 win shares).
Any real model has to beat this.

- **MAE** (mean absolute error): how many win shares a prediction is off by, on average.
- **R2**: how well my machine learning model explains or predicts the variation in the target variable

**Result:** MAE 5.90, R2 0.00.

In [ ]:
average = train["ws_first4"].mean()
print("AVERAGE(train):",average)

predicted = np.full(len(test), average)
real = test["ws_first4"]
mae = mean_absolute_error(real, predicted)
r2 = r2_score(real, predicted)

print("MAE:", mae)
print("R2:", r2)


AVERAGE(train): 6.13851030110935
MAE: 5.8981718832540935
R2: -0.002908151897184963


## Predicting `ws_first4` (a number)

The `evaluate` function trains a model on the training drafts and scores it on the test drafts,
so each model takes one line.

| Model | MAE | R² |
|---|---|---|
| Baseline (guess the average) | 5.90 | 0.00 |
| Pick only (linear) | 4.73 | 0.221 |
| Log pick (linear) | 4.55 | 0.215 |
| Log pick + no college | 4.52 | 0.218 |
| Log pick + no college + draft year | 4.77 | 0.204 |
| Gradient boosting | **4.30** | **0.233** |

**Findings**
- **The pick number explains about 22%** of early-career success, and cuts the error from 5.90 to 4.73.
- **`log_pick` helps a little** (4.73 → 4.55), because it matches the curved shape of the data.
- **`no_college` barely helps** (4.55 → 4.52). Only 309 of 1,742 picks are no-college, and the effect is small.
- **`draft_year` makes the linear model worse.** It learns a straight-line trend over 1989–2010 and extends it
  into years it never saw (**extrapolation**), but notebook 03 showed no steady trend over time.
- **Gradient boosting is the best, but only slightly.** It is built from many small "if pick ≤ X" rules, so it can
  follow curves and does not extend trends. The trade-off: it is more accurate but harder to explain than one formula.

In [6]:
def evaluate(name, features, model):
    model.fit(train[features], train["ws_first4"])
    predictions = model.predict(test[features])

    mae = mean_absolute_error(test["ws_first4"], predictions)
    r2 = r2_score(test["ws_first4"], predictions)
    print(f"{name:30} MAE: {mae} R2: {r2}")

evaluate("Pick only (linear)", ["overall_pick"], LinearRegression())
evaluate("Log pick (linear)", ["log_pick"], LinearRegression())
evaluate("Log pick + no college", ["log_pick", "no_college"], LinearRegression())
evaluate("Log pick + no college + year", ["log_pick", "no_college", "draft_year"], LinearRegression())
evaluate("Gradient boosting", ["overall_pick", "no_college", "draft_year"],
         GradientBoostingRegressor(max_depth=2, n_estimators=200, learning_rate=0.05, random_state=0))

Pick only (linear)             MAE: 4.728151922716685 R2: 0.22101398524973292
Log pick (linear)              MAE: 4.553236639829111 R2: 0.21534621571333568
Log pick + no college          MAE: 4.51691169578798 R2: 0.21847849222230553
Log pick + no college + year   MAE: 4.773017866222504 R2: 0.20386906216441858
Gradient boosting              MAE: 4.300696512992535 R2: 0.23311376807612683


## Predicting `hit` (yes/no): why not accuracy?

Only 22% of test picks are hits. A "model" that always says "not a hit" is **78% accurate** but finds zero hits.
When one answer is much more common (**class imbalance**), accuracy rewards guessing the common answer.

So I use **AUC** instead: how often the model gives a real hit a higher probability than a non-hit.
0.5 is a coin flip, 1.0 is perfect.

In [7]:
test_hit_rate = test["hit"].mean()
print(f"Test hit rate: {test_hit_rate:.0%}")
print(f"Accuracy of always guessing 'not a hit': {1 - test_hit_rate:.0%}")

Test hit rate: 22%
Accuracy of always guessing 'not a hit': 78%


## Classification models

`evaluate_hit` works like `evaluate`, but predicts the **probability** of being a hit and scores it with AUC.
For yes/no targets I use classifiers (`LogisticRegression`, `GradientBoostingClassifier`) instead of regressors.

**Findings:** AUC 0.774 (log pick), 0.778 (+ no college), 0.781 (gradient boosting).
The same story as the regression: the pick does almost all the work, and extra features add very little.

In [8]:
def evaluate_hit(name, features, model, target):
    model.fit(train[features], train[target])
    probabilities = model.predict_proba(test[features])[:, 1]
    auc = roc_auc_score(test[target], probabilities)
    print(f"{name:30} AUC {auc:.3f}")

evaluate_hit("Log pick", ["log_pick"], LogisticRegression(),"hit")
evaluate_hit("Log pick + no college", ["log_pick", "no_college"], LogisticRegression(),"hit")
evaluate_hit("Gradient boosting", ["overall_pick", "no_college", "draft_year"],
            GradientBoostingClassifier(max_depth=2, n_estimators=200, learning_rate=0.05, random_state=0),"hit")


Log pick                       AUC 0.774
Log pick + no college          AUC 0.778
Gradient boosting              AUC 0.781


## Sensitivity check: does the hit threshold matter?

In notebook 02 I chose 10 win shares as the hit threshold, and said it was a judgment call.
Here I rerun the same model with thresholds of 8 and 15.

Note: I recreate `train` and `test` before adding the new columns, so the columns are not lost.

In [15]:
train = players[players["draft_year"] <= LAST_TRAIN_YEAR].copy()
test = players[players["draft_year"] > LAST_TRAIN_YEAR].copy()

train["hit_8"] = train["ws_first4"] >=8
test["hit_8"] = test["ws_first4"] >=8
train["hit_15"] = train["ws_first4"] >=15
test["hit_15"] = test["ws_first4"] >=15

print(test["hit_8"].mean())
print(test["hit"].mean())
print(test["hit_15"].mean())

0.2916666666666667
0.21875
0.1125


In [10]:
evaluate_hit("Hit ≥ 8", ["log_pick", "no_college"], LogisticRegression(), "hit_8")
evaluate_hit("Hit ≥ 10", ["log_pick", "no_college"], LogisticRegression(), "hit")
evaluate_hit("Hit ≥ 15", ["log_pick", "no_college"], LogisticRegression(), "hit_15")

Hit ≥ 8                        AUC 0.794
Hit ≥ 10                       AUC 0.778
Hit ≥ 15                       AUC 0.742


**Findings**

| Hit | Test hit rate | AUC |
|---|---|---|
| ≥ 8 win shares | 29% | 0.794 |
| ≥ 10 win shares | 22% | 0.778 |
| ≥ 15 win shares | 11% | 0.742 |

- Useful players are easier to predict than near-stars. Stars are rarer, and what makes them is harder to see on draft night.
- At every threshold the model is clearly better than a coin flip and far from perfect, so the conclusion
  does not depend on choosing 10. The result is **robust**.

## Answer and summary

**Question:** using only what a team knows on draft night, how well can I predict a player's early career?

**Answer:** moderately well, and almost entirely through the pick number.
- **Win shares (a number):** the best model is off by about **4.3 win shares** per player (MAE), compared with 5.9 for
  guessing the average. It explains about **23%** of the differences between players (R² 0.23).
- **Hit or not (yes/no):** the model ranks a real hit above a non-hit about **78%** of the time (AUC 0.78).
  A coin flip would be 50%; a perfect model 100%.
- So draft-night information gives a clear but limited edge: most of what makes a player succeed can't be seen on draft night.

**Summary**
1. **The pick number explains about 22%** of early-career success. Draft order carries real information.
2. **Other draft night facts add almost nothing.** College background helps slightly; draft year hurts the linear model.
3. **A flexible model (gradient boosting) is only slightly better** than a simple one.
4. **About 78% cannot be predicted** from draft-night data: development, injuries, fit and luck.
5. Compared with my original version: **R² 0.94 → 0.22.** The new number is lower, but it is honest, because it only
   uses what a team knows on draft night.

**Next (notebook 05):** players who did far better or worse than their draft slot predicted are the steals and busts.